In [ ]:
from pathlib import Path
import random
import pickle

import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Border, Side, Font, Alignment
from openpyxl.utils import get_column_letter


In [ ]:
SEED = 1128
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
real = pd.read_csv(
    Path.home()
    / "Research"
    / "Virtual-ALS-Patients"
    / "data"
    / "no_labels_no_birth_year"
    / "train.csv"
)
real.head()

In [ ]:
# filter out patients with no temporal data (all unknowns)

dfs = []

for _, pdf in real.groupby(by="REF"):
    if (pdf[[f"P{j}" for j in range(1, 13)]] == -1.0).all().all():
        continue
    dfs.append(pdf)
    
real = pd.concat(dfs, ignore_index=True)

In [ ]:
version = "2025-03-28 15:06:33"
fname = f"synthetic_dfs_n_samples=100_patientflow_{version}_seed=42.pkl"

with open(fname, "rb") as f:
    gen_dfs, categorical_feats_info = pickle.load(f)

In [ ]:
with open(f"rule_checks_historical_min_n_samples=100_patientflow_{version}_seed=42.pkl", "rb") as f:
    rule_checks = pickle.load(f)

In [ ]:
eligible_real_patient_refs = []
for _, pdf in real.groupby(by="REF"):
    if len(pdf) > 1:
        eligible_real_patient_refs.append(pdf["REF"].iloc[0])

In [ ]:
real_patient_ids = np.random.choice(eligible_real_patient_refs, size=12, replace=False)
real_patient_ids

In [ ]:
random_real_patients = pd.concat([real[real["REF"] == pid] for pid in real_patient_ids], ignore_index=True)
random_real_patients

In [ ]:
random_real_patients_dfs = []

for _, pdf in random_real_patients.groupby(by="REF"):
    pdf = pdf.reset_index(drop=True)
    pdf["medianDate"] = pd.to_datetime(pdf["medianDate"], format="%Y-%m-%d")
    original_dates = pdf["medianDate"].copy()
                
    # Calculate days between consecutive visits
    for i in range(1, len(pdf)):
        pdf.loc[i, "medianDate"] = int((original_dates[i] - original_dates[i-1]).days)
    pdf.loc[0, "medianDate"] = 0
    random_real_patients_dfs.append(pdf)

random_real_patients = pd.concat(random_real_patients_dfs, ignore_index=True)

In [ ]:
random_real_patients["is_real"] = True

In [ ]:
def len_more_than_one(gen_df):
    correct_patients = np.ones(len(gen_df["REF"].unique()), dtype=np.bool8)

    for pid, (_, pdf) in enumerate(gen_df.groupby(by="REF")):
        # have to check if the values are always increasing
        correct_patients[pid] = len(pdf) > 1
    
    return correct_patients

In [ ]:
def no_zero_delta_but_first(gen_df):
    correct_patients = np.ones(len(gen_df["REF"].unique()), dtype=np.bool8)

    for pid, (_, pdf) in enumerate(gen_df.groupby(by="REF")):
        # medianDate from index 1 to the end must be above 0.0
        correct_patients[pid] = pdf["medianDate"].iloc[1:].min() > 0.0
    
    return correct_patients

In [ ]:
len_rule = np.array([len_more_than_one(gen_df) for gen_df in gen_dfs])

In [ ]:
delta_rule = np.array([no_zero_delta_but_first(gen_df) for gen_df in gen_dfs])

In [ ]:
eligible_fake_patients = rule_checks[:, 0] & rule_checks[:, 1] & len_rule & delta_rule
eligible_fake_patients

In [ ]:
eligible_fake_patients.shape

In [ ]:
true_indices = np.where(eligible_fake_patients)
true_pairs = np.column_stack(true_indices)
true_pairs

In [ ]:
len(true_pairs)

In [ ]:
# different sampling to make patient lengths more balanced
true_pairs_with_lengths = []
true_pairs_lens_set = set()

for i, j in true_pairs:
    true_pairs_lens_set.add(len(gen_dfs[i][gen_dfs[i]["REF"] == j]))
    true_pairs_with_lengths.append((i, j, len(gen_dfs[i][gen_dfs[i]["REF"] == j])))

true_pairs_with_lengths = np.array(true_pairs_with_lengths)

In [ ]:
real_seq_lens = [len(pdf) for _, pdf in real.groupby(by="REF")]

seq_distribution = np.ones(max(real_seq_lens) + 1)

for idx, count in zip(*np.unique(real_seq_lens, return_counts=True)):
    seq_distribution[idx] = count

seq_distribution[0] = 0.0
for i in range(1, len(seq_distribution)):
    if i not in true_pairs_lens_set:
        seq_distribution[i] = 0.0
seq_distribution = (
    seq_distribution / seq_distribution.sum()
)

print(seq_distribution)
random_lengths = [np.random.multinomial(1, seq_distribution).argmax() for _ in range(12)]
print(random_lengths)

# random_lengths = np.random.choice(np.unique(true_pairs_with_lengths[:, 2]), size=12, replace=True)
random_fake_patients_ids = []

for length in random_lengths:
    candidates = true_pairs_with_lengths[true_pairs_with_lengths[:, 2] == length]
    random_fake_patients_ids.append(candidates[np.random.choice(len(candidates))])

random_fake_patients_ids

In [ ]:
# random_fake_patients_ids = true_pairs[np.random.choice(len(true_pairs), size=12, replace=False)]
# random_fake_patients_ids

In [ ]:
random_fake_patients = []

for i,j,_ in random_fake_patients_ids:
    pdf = gen_dfs[i][gen_dfs[i]["REF"] == j]
    pdf = pdf.reset_index(drop=True)
    pdf["medianDate"] = pdf["medianDate"].astype(int)
    pdf.loc[0, "medianDate"] = 0
    random_fake_patients.append(pdf)

random_fake_patients = pd.concat(random_fake_patients, ignore_index=True)
random_fake_patients

In [ ]:
random_fake_patients["is_real"] = False

In [ ]:
random_fake_patients["Age_onset"] = random_fake_patients["Age_onset"].apply(lambda x: np.floor(x))

In [ ]:
random_patients = []

for _, pdf in list(random_real_patients.groupby(by="REF")) + list(random_fake_patients.groupby(by="REF")):
    pdf = pdf.reset_index(drop=True)
    random_patients.append(pdf)

random.shuffle(random_patients)

for i, pdf in enumerate(random_patients, start=1):
    pdf["REF"] = i

random_patients = pd.concat(random_patients, ignore_index=True)
random_patients

In [ ]:
random_patients["Height (m)"] = random_patients["Height (m)"].apply(lambda x: np.round(x, 2))
random_patients["Weight"] = random_patients["Weight"].apply(lambda x: np.round(x, 2))

In [ ]:
random_real_patients.to_csv("random_real_patients.csv", index=False)

In [ ]:
random_patients = random_patients.drop(columns=["prog_profile"]) #"medianDate"])
random_patients

In [ ]:
random_patients[random_patients["is_real"] == False]

In [ ]:
len(random_patients[random_patients["is_real"] == True].groupby(by="REF"))

In [ ]:
len(random_patients[random_patients["is_real"] == False].groupby(by="REF"))

In [ ]:
random_patients.to_csv("random_patients_with_labels.csv", index=False)

In [ ]:
random_patients = random_patients.drop(columns=["is_real"])
random_patients.to_csv("random_patients_without_labels.csv", index=False)

In [ ]:
# Create a workbook
wb = Workbook()
ws = wb.active
ws.title = "Patient Evaluation"
ws.sheet_view.showGridLines = False

# Define styles
thin_border = Border(left=Side(style='thin'), right=Side(style='thin'), 
                     top=Side(style='thin'), bottom=Side(style='thin'))
thick_border = Border(
    left=Side(style='thick'), 
    right=Side(style='thick'), 
    top=Side(style='thick'), 
    bottom=Side(style='thick')
)
header_border = Border(bottom=Side(style='thick'))
no_border = Border()
bottom_border = Border(bottom=Side(style='medium'))


headers = ['REF', 'Age onset', 'Gender', 'Ethnicity', 'NIV', 'Tracheostomy', 'PEG', 'UMNvsLMN', 'Onset', 
           'Limb_O', 'Limbs Impairment', 'Limbs Side', 'Height (m)', 'Weight', 'Weightloss_>10%',
           'ALS familiar history', 'Ever smoked', 'Blood hypertension', 'Diabetes', 'Dyslipidemia',
           'Thyroid', 'Autoimmune', 'Stroke', 'Cardiac Disease', 'Primary Cancer', 
           'SOD1 Mutation', 'C9orf72', 'TARDBP mutation', 'FUS mutation']

headers_to_col_mapping = {
    "Age onset": "Age_onset",
    "Limbs Impairment": "Limbs_Impairment",
    "Limbs Side": "Limbs_Side",
    "ALS familiar history": "ALS_familiar_history",
    "Ever smoked": "Ever_smoked",
    "Blood hypertension": "Blood_hypertension",
    "Cardiac Disease": "Cardiac_disease", 
    "Primary Cancer": "Primary_cancer",
    "Time (days) since previous visit": "medianDate",
}

# Add P1-P12 (observations through time)
for i in range(1, 13):
    headers.append(f'P{i}')

# Add the time column after P12
headers.append("Time (days) since previous visit")

default_font = Font(name="Aptos Narrow (Body)", size=14)
default_font_bold = Font(name="Aptos Narrow (Body)", size=14, bold=True)

# Define a function to write headers
def write_headers(start_row):
    for col_idx, header in enumerate(headers, 1):
        cell = ws.cell(row=start_row, column=col_idx)
        cell.value = header
        cell.border = header_border
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.font = default_font
    
    return start_row + 1

# Current row to write data
current_row = 1

# Process each patient
for _, patient in random_patients.groupby(by="REF"):
    patient = patient.reset_index(drop=True)
    patient_id = patient.loc[0, 'REF'] 
    
    num_observations = len(patient)
    
    # Write headers for each patient
    current_row = write_headers(current_row)
    
    patient_start_row = current_row
    
    ws.cell(row=current_row, column=1).value = patient_id
    ws.cell(row=current_row, column=1).font = default_font
    
    # Fill patient data (columns 2 to 29)
    for col_idx, field in enumerate(headers[1:29], 2):
        if field in headers_to_col_mapping:
            field = headers_to_col_mapping[field]
        
        try:
            ws.cell(row=current_row, column=col_idx).value = patient.loc[0, field]
            ws.cell(row=current_row, column=col_idx).font = default_font
        except KeyError:
            # Handle missing columns in the data
            pass
        
        # Add styling to all cells - no vertical borders for static features
        ws.cell(row=current_row, column=col_idx).alignment = Alignment(horizontal='center', vertical='center')
        ws.cell(row=current_row, column=col_idx).border = no_border
        
    # Create observation lines (P1-P12 and Time column)
    for p_idx in range(num_observations):
        # Fill observation data
        for col_idx in range(30, len(headers) + 1):  # P1-P12 columns + Time column
            # Get the column name
            p_col = headers[col_idx-1]
            
            if p_col in headers_to_col_mapping:
                col_name = headers_to_col_mapping[p_col]
            else:
                col_name = p_col
            
            try:
                # For P1-P12 columns, we use their name directly
                if p_col.startswith('P'):
                    value = patient[p_col].iloc[p_idx] if p_idx < len(patient) and p_col in patient.columns else None
                else:
                    # For the time column or other columns
                    value = patient[col_name].iloc[p_idx] if p_idx < len(patient) and col_name in patient.columns else None
            except (KeyError, IndexError):
                value = None
            
            ws.cell(row=current_row + p_idx, column=col_idx).value = value
            ws.cell(row=current_row + p_idx, column=col_idx).alignment = Alignment(horizontal='center', vertical='center')
            ws.cell(row=current_row + p_idx, column=col_idx).border = Border(bottom=Side(style='thick'))
            ws.cell(row=current_row + p_idx, column=col_idx).font = default_font
    
    # Add bottom border to the last row of data
    last_data_row = current_row + max(0, num_observations - 1)
    for col_idx in range(1, len(headers) + 1):
        ws.cell(row=last_data_row, column=col_idx).border = bottom_border
    
    # Update current_row to account for all observations
    current_row += max(1, num_observations)
    
    # Merge cells for patient data if there are multiple observations
    if num_observations > 1:
        for col_idx in range(1, 30):  # Only merge the patient info columns, not the P1-P12 or Time columns
            ws.merge_cells(
                start_row=patient_start_row, 
                start_column=col_idx, 
                end_row=patient_start_row + num_observations - 1, 
                end_column=col_idx
            )
            # Center align merged cells
            merged_cell = ws.cell(row=patient_start_row, column=col_idx)
            merged_cell.alignment = Alignment(horizontal='center', vertical='center')

    # Add spacing row after patient data
    current_row += 1
    
    # Add a separator line with "Questions:" label (moved one column to the right)
    ws.cell(row=current_row, column=1).value = "Questions:"
    ws.cell(row=current_row, column=1).font = default_font_bold
    current_row += 1
    
    # Q1: Real or Synthetic
    q1_row = current_row
    ws.cell(row=q1_row, column=2).value = f"Q1. Does Patient {patient_id} appear to be Real or Synthetic? (place X for answer)"
    ws.cell(row=q1_row, column=2).font = default_font
    ws.merge_cells(start_row=q1_row, start_column=2, end_row=q1_row, end_column=8)
    current_row += 1
    
    # Real checkbox
    ws.cell(row=current_row, column=4).value = "Real"
    ws.cell(row=current_row, column=4).font = default_font
    # Create checkbox border
    checkbox_cell = ws.cell(row=current_row, column=3)
    checkbox_cell.border = thick_border
    current_row += 1

    ws.row_dimensions[current_row].height = 6
    current_row += 1
    
    # Synthetic checkbox
    ws.cell(row=current_row, column=4).value = "Synthetic"
    ws.cell(row=current_row, column=4).font = default_font
    # Create checkbox border
    checkbox_cell = ws.cell(row=current_row, column=3)
    checkbox_cell.border = thick_border
    current_row += 1
    
    # Add a blank line
    current_row += 1
    
    # Q2: Confidence
    q2_row = current_row
    ws.cell(row=q2_row, column=2).value = "Q2. Confidence in previous answer? (place X for answer)"
    ws.merge_cells(start_row=q2_row, start_column=2, end_row=q2_row, end_column=8)
    ws.cell(row=q2_row, column=2).font = default_font
    current_row += 2
    
    # Confidence scale labels
    ws.cell(row=current_row, column=2).value = "Not confident at all"
    ws.cell(row=current_row, column=2).font = default_font
    ws.cell(row=current_row, column=7).value = "Completely confident"
    ws.cell(row=current_row, column=7).font = default_font
    current_row += 1
    
    # Scale numbers 1-5 with checkbox borders
    for i in range(3, 8):
        ws.cell(row=current_row, column=i).border = thick_border
    
    current_row += 1
    ws.cell(row=current_row, column=3).value = "1"
    ws.cell(row=current_row, column=3).font = default_font
    ws.cell(row=current_row, column=3).alignment = Alignment(horizontal='right')

    ws.cell(row=current_row, column=7).value = "5"
    ws.cell(row=current_row, column=7).font = default_font
    ws.cell(row=current_row, column=7).alignment = Alignment(horizontal='right')
    
    current_row += 1
    
    # Add a blank line
    current_row += 1
    
    # Q3: Reasoning
    q3_row = current_row
    ws.cell(row=q3_row, column=2).value = "Q3. Reasoning behind Q1's Answer?"
    ws.merge_cells(start_row=q3_row, start_column=2, end_row=q3_row, end_column=8)
    ws.cell(row=q3_row, column=2).font = default_font
    current_row += 1
    
    # Note for Q3
    note_row = current_row
    ws.cell(row=note_row, column=3).value = "Note: If the answer to question 1 (Q1)  was \"Synthetic\", then a response is required. If the answer was \"Real\", a response is only required if your confidence level was below 3."
    ws.cell(row=note_row, column=3).font = default_font
    ws.cell(row=note_row, column=3).alignment = Alignment(horizontal='center', vertical='top')
    ws.merge_cells(start_row=note_row, start_column=3, end_row=note_row, end_column=16)
    current_row += 1
    
    # Answer field
    answer_row = current_row
    ws.cell(row=answer_row, column=3).value = "Answer:"
    ws.cell(row=answer_row, column=3).font = default_font
    ws.merge_cells(start_row=answer_row, start_column=4, end_row=answer_row + 1, end_column=16)
    ws.cell(row=answer_row, column=4).alignment = Alignment(vertical='top', horizontal='left', wrap_text=True)
    ws.cell(row=answer_row, column=4).font = default_font
    
    for r in range(answer_row, answer_row + 2):  # +2 because end_row is inclusive
        for c in range(4, 17):  # +1 because end_column is inclusive
            # Apply thick border to outer edges only
            border_left = Side(style='thick') if c == 4 else Side(style='none')
            border_right = Side(style='thick') if c == 16 else Side(style='none')
            border_top = Side(style='thick') if r == answer_row else Side(style='none')
            border_bottom = Side(style='thick') if r == answer_row + 1 else Side(style='none')
            
            ws.cell(row=r, column=c).border = Border(
                left=border_left,
                right=border_right,
                top=border_top,
                bottom=border_bottom
            )

    current_row += 3  # Add extra space before next patient

# Format the worksheet
col_width = [
    10.50, # REF
    5.83, # Age onset
    8.83,
    10.0,
    7.50,
    13.33,
    10.0,
    11.50,
    12.17,
    10.0,
    11.50,
    10.0,
    10.0,
    10.0,
    17.17,
    16.33,
    9.83,
    13.17,
    10.0,
    12.33,
    10.0,
    12.17,
    10.0,
    10.0,
    10.50,
    11.83,
    10.0,
    11.33,
    10.0,
] + [10.0] * 13

for col in range(1, len(headers) + 1):
    ws.column_dimensions[get_column_letter(col)].width = col_width[col-1] * 1.08

# No freeze panes since there's no single header row anymore

# Save the workbook
wb.save("lisbon_dataset_evaluation.xlsx")